# 🏏 IPL Feature Engineering Pipeline

Welcome to the **Feature Engineering** notebook for the **IPL Win Probability Predictor**. Feature engineering is the cornerstone of building predictive classifiers in cricket. It translates raw ball-by-ball actions (runs scored, wickets lost, overs bowled) into high-value situational indicators that represent game pressure and momentum.

### 🎯 Objectives
In this notebook, we will build the core data pipeline that extracts and constructs our features:
1. **`current_score`**: Cumulative runs chased at any specific ball.
2. **`target`**: Target runs to chase (First innings total + 1).
3. **`runs_left`**: Difference between target and current score.
4. **`balls_left`**: Remaining legal balls out of 120.
5. **`crr` (Current Run Rate)**: Scaled runs scored per over.
6. **`rrr` (Required Run Rate)**: Scaled runs required per remaining overs.
7. **`wickets_remaining`**: Remaining wickets (10 - wickets lost).
8. **`match_pressure_index` (MPI)**: Custom complex metric reflecting stress level.
9. **`required_boundary_percentage` (RBP)** & **`run_momentum` (RM)**: Boundary pressure and momentum indexes.
10. **Save processed dataset**: Persist `training_data.csv` for downstream model training.

---  
## ⚙️ 1. Setup & Ingestion
First, we load our cleaned intermediate datasets created during preprocessing.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project directories are visible
sys.path.append(os.path.abspath(".."))
import config

processed_matches_path = os.path.join("..", "data", "processed", "matches_cleaned.csv")
processed_deliveries_path = os.path.join("..", "data", "processed", "deliveries_cleaned.csv")

# Verify paths exist before proceeding
if not os.path.exists(processed_matches_path) or not os.path.exists(processed_deliveries_path):
    print("⚠️ Processed clean datasets not found! Invoking pipeline initialization...")
    import train
    train.verify_project_structure()
    train.generate_synthetic_raw_data()
    # Standardize data paths relative to the notebooks workspace directory
    processed_matches_path = os.path.join("data", "processed", "matches_cleaned.csv")
    processed_deliveries_path = os.path.join("data", "processed", "deliveries_cleaned.csv")

matches_df = pd.read_csv(processed_matches_path)
deliveries_df = pd.read_csv(processed_deliveries_path)

print(f"✅ Matches dataset loaded: {len(matches_df)} rows")
print(f"✅ Deliveries dataset loaded: {len(deliveries_df)} rows")

---  
## 🎯 2. Target Formulation
To formulate a run chase, we must compute the exact target set by the team batting first. The target is defined as **`First Innings Total Score + 1`**.

In [ ]:
print("Calculating 1st innings total runs...")
# Sum up the total runs (including extras) grouped by match_id for the first innings (inning == 1)
total_score_df = deliveries_df[deliveries_df["inning"] == 1].groupby("match_id")["total_runs"].sum().reset_index()
total_score_df.rename(columns={"total_runs": "first_innings_score"}, inplace=True)

# Target score is 1st innings total + 1
total_score_df["target"] = total_score_df["first_innings_score"] + 1

print("Sample target records computed:")
display(total_score_df.head())

# Merge target scores back into matches metadata
matches_df = matches_df.merge(total_score_df, left_on="id", right_on="match_id", how="inner")

---  
## 🏏 3. Isolating the Run Chase (Second Innings)
Since win probability models analyze the progression of a chase, we isolate all deliveries belonging to the second innings (`inning == 2`).

In [ ]:
chase_deliveries = deliveries_df[deliveries_df["inning"] == 2].copy()

# Merge important match features (target, winner, venue, city) into deliveries
matches_subset = matches_df[["id_x", "target", "winner", "venue", "city"]].copy().rename(columns={"id_x": "id"})
chase_deliveries = chase_deliveries.merge(matches_subset, left_on="match_id", right_on="id", how="inner")

print(f"Chasing deliveries isolated. Total chasing logs: {len(chase_deliveries)}")

---  
## 📈 4. Calculating Runs Left & Balls Left
For any specific delivery during a run chase, we calculate:
* **`current_score`**: Cumulative runs chased so far within that match.
* **`runs_left`**: `Target - Current Score` (clamped at a minimum of 0).
* **`balls_left`**: `120 - Balls Bowled` (clamped at a minimum of 0).

In [ ]:
# Compute current score (cumulative runs) per match
chase_deliveries["current_score"] = chase_deliveries.groupby("match_id")["total_runs"].cumsum()

# Compute runs left to chase
chase_deliveries["runs_left"] = (chase_deliveries["target"] - chase_deliveries["current_score"]).clip(lower=0)

# Compute balls left
# Check if the dataset over values are 1-indexed (1-20) or 0-indexed (0-19)
min_over = chase_deliveries["over"].min()
if min_over == 1:
    balls_bowled = (chase_deliveries["over" ] - 1) * 6 + chase_deliveries["ball"]
else:
    balls_bowled = chase_deliveries["over"] * 6 + chase_deliveries["ball"]

chase_deliveries["balls_left"] = (120 - balls_bowled).clip(lower=0)

print("Sample of computed scores, runs, and balls remaining:")
display(chase_deliveries[["match_id", "over", "ball", "total_runs", "current_score", "runs_left", "balls_left"]].head())

---  
## 🚪 5. Wickets Remaining
Wickets are the primary constraint in a run chase. We compute the cumulative wickets lost and subtract this value from the maximum available wickets (10) to determine **`wickets_remaining`**.

In [ ]:
# Identify deliveries where a batsman was dismissed
chase_deliveries["is_wicket"] = chase_deliveries["player_dismissed"].notna().astype(int)

# Cumulative sum of wickets lost per match
chase_deliveries["wickets_down"] = chase_deliveries.groupby("match_id")["is_wicket"].cumsum()

# Calculate wickets remaining (clamped between 0 and 10)
chase_deliveries["wickets_remaining"] = (10 - chase_deliveries["wickets_down"]).clip(0, 10)

print("Wickets tracking samples:")
display(chase_deliveries[["match_id", "is_wicket", "wickets_down", "wickets_remaining"]].head())

---  
## ⚡ 6. Current Run Rate (CRR) & Required Run Rate (RRR)
Run rates standardize speed scales over the course of the chase:
* **CRR**: `(Current Score * 6) / Balls Bowled`
* **RRR**: `(Runs Left * 6) / Balls Left`

In [ ]:
# Avoid division by zero at the start of the second innings
safe_balls_bowled = np.where(balls_bowled == 0, 1, balls_bowled)
chase_deliveries["crr"] = np.round((chase_deliveries["current_score"] * 6.0) / safe_balls_bowled, 2)

# Avoid division by zero at the final ball or death overs
safe_balls_left = np.where(chase_deliveries["balls_left"] == 0, 1, chase_deliveries["balls_left"])
chase_deliveries["rrr"] = np.round((chase_deliveries["runs_left"] * 6.0) / safe_balls_left, 2)

# Apply logic for extreme boundary conditions:
# 1. If balls_left is 0 and runs_left > 0, set RRR to high fallback (99.99)
chase_deliveries.loc[(chase_deliveries["balls_left"] == 0) & (chase_deliveries["runs_left"] > 0), "rrr"] = 99.99
# 2. If target has already been reached, RRR should be 0.0
chase_deliveries.loc[chase_deliveries["runs_left"] <= 0, "rrr"] = 0.0

print("CRR and RRR updates preview:")
display(chase_deliveries[["current_score", "crr", "runs_left", "balls_left", "rrr"]].head())

---  
## 🌋 7. Match Pressure Index (MPI) & Advanced Sports Metrics
Standard cricket metrics (CRR, RRR) alone do not capture the compounding psychological pressure during the final overs of a match. We construct **`match_pressure_index` (MPI)** to combine these dynamics under a unified non-linear index.

$$MPI = \left( \frac{RRR}{CRR + 0.1} \right) \times \left( \frac{10}{Wickets Remaining + 0.1} \right) \times Progression$$

Where *Progression* is $\frac{Balls Bowled}{120}$.

In [ ]:
# Progression represents match stage
progression = balls_bowled / 120.0

# Compute custom Match Pressure Index (MPI) and clamp output between 0 and 100
chase_deliveries["match_pressure_index"] = (
    (chase_deliveries["rrr"] / (chase_deliveries["crr"] + 0.1)) *
    (10.0 / (chase_deliveries["wickets_remaining"] + 0.1)) *
    progression
)
chase_deliveries["match_pressure_index"] = np.round(chase_deliveries["match_pressure_index"].clip(0, 100), 2)

# Required Boundary Percentage: The percentage of remaining runs that must come from boundaries to maintain structural momentum
chase_deliveries["required_boundary_percentage"] = np.round((chase_deliveries["rrr"] / 24.0) * 100.0, 2).clip(0, 100)

# Run Momentum: CRR margin relative to RRR
chase_deliveries["run_momentum"] = np.round(chase_deliveries["crr"] - chase_deliveries["rrr"], 2)

print("Advanced sports metrics engineered:")
display(chase_deliveries[["rrr", "crr", "wickets_remaining", "match_pressure_index", "required_boundary_percentage", "run_momentum"]].head())

---  
## 💾 8. Label Formatting & Saving Dataset
We define the target learning label **`result`**: did the chasing team successfully chase the target runs and win the match (`winner == batting_team`)?

In [ ]:
# Generate learning label
chase_deliveries["result"] = (chase_deliveries["winner"] == chase_deliveries["batting_team"]).astype(int)

# Collect the finalized feature space set for serialization
final_features = [
    "match_id", "batting_team", "bowling_team", "venue", "city",
    "target", "current_score", "runs_left", "balls_left", "wickets_remaining",
    "crr", "rrr", "match_pressure_index", "required_boundary_percentage", "run_momentum", "result"
]
training_df = chase_deliveries[final_features].copy()

display(training_df.head())

# Define processed storage directory path
os.makedirs(os.path.join("..", "data", "processed"), exist_ok=True)
training_data_path = os.path.join("..", "data", "processed", "training_data.csv")

# Save training dataframe
training_df.to_csv(training_data_path, index=False)
print(f"💾 Successfully saved cleaned training features ({len(training_df)} rows) to: {training_data_path}")

## 🎓 Summary Inferences
By translating raw, discrete, ball-by-ball actions into high-value situational features like **`rrr`**, **`match_pressure_index`**, and **`run_momentum`**, we have equipped our machine learning estimators with the context required to output highly accurate, real-time win probabilities.